In [2]:
import numpy as np
import pandas as pd
import plotly.offline as pyo
import plotly.graph_objects as go

# Plotly Graph Objects

In [7]:
match=pd.read_csv('matches.csv')
delivery=pd.read_csv('deliveries.csv')

ipl=delivery.merge(match,left_on='match_id',right_on='id')

In [7]:
ipl.head(1)

,match_id,inning,batting_team,bowling_team,over,ball,batter,bowler,non_striker,batsman_runs,...,toss_decision,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2
0,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,SC Ganguly,P Kumar,BB McCullum,0,...,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen


# 1. Scatter Plot

Plotting scatter graph for top 50 batsman(Avg vs Strike Rate)

In [8]:
top50=ipl.groupby('batter')['batsman_runs'].sum().sort_values(ascending=False).head(50).index
new_ipl=ipl[ipl['batter'].isin(top50)]
new_ipl.head(1)


,match_id,inning,batting_team,bowling_team,over,ball,batter,bowler,non_striker,batsman_runs,...,toss_decision,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2
1,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,BB McCullum,P Kumar,SC Ganguly,0,...,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen


In [9]:
# Calculating SR
totalruns=new_ipl.groupby('batter')['batsman_runs'].sum()
totalballs=new_ipl.groupby('batter')['batsman_runs'].count()
sr=(totalruns/totalballs)*100
sr=sr.reset_index()

In [10]:
# Calculating Avg
out=ipl[ipl['player_dismissed'].isin(top50)]
nouts=out['player_dismissed'].value_counts()

avg=totalballs/nouts
avg=avg.reset_index()
avg.rename(columns={0:'avg'},inplace=True)
sr=sr.rename(columns={'batsman_runs':'Strike Rate'})

In [11]:
new_df=avg.merge(sr,on='batter')

In [57]:
# Plotting Graph
traces=go.Scatter(x=new_df['avg'],y=new_df['Strike Rate'],mode='markers',text=new_df['batter'],
                  marker={'color':'#00a65a','size':12})

data=[traces]
layout=go.Layout(title='Batsman Avg vs Strike rate',xaxis={'title':'Avg'},yaxis={'title':'Strike Rate'})

fig=go.Figure(data=data,layout=layout)
pyo.plot(fig,filename='myfile.html')

'myfile.html'

# 2. Line Chart

Year by year performance of a batsman

In [ ]:
vk=ipl[ipl['batter']=='V Kohli']
vkdf=vk.groupby('season')['batsman_runs'].sum().reset_index()

md=ipl[ipl['batter']=='MS Dhoni']
mddf=md.groupby('season')['batsman_runs'].sum().reset_index()

,season,batsman_runs
0,2007/08,165
1,2009,246
2,2009/10,307
3,2011,557
4,2012,364
5,2013,639
6,2014,359
7,2015,505
8,2016,973
9,2017,308


In [71]:
trace=go.Scatter(x=vkdf['season'],y=vkdf['batsman_runs'],mode='lines+markers',marker={'color':'#00a65a'},name='Virat Kohli')
trace1=go.Scatter(x=mddf['season'],y=mddf['batsman_runs'],mode='lines+markers',name='MS Dhoni')

data1=[trace,trace1]

layout1=go.Layout(title="Batsman's Performance",xaxis={'title':'Season'},yaxis={'title':'Runs'})


fig=go.Figure(data=data1,layout=layout1)
pyo.plot(fig)

'temp-plot.html'

In [ ]:
# Multiple line chart
def batsman_comp(*name):
    data=[]
    for i in name:
        batter=ipl[ipl['batter']==i]
        performance=batter.groupby('season')['batsman_runs'].sum().reset_index()
        trace=go.Scatter(x=performance['season'],y=performance['batsman_runs'],mode='lines+markers',name=i)

        data.append(trace)

    layout=go.Layout(title='Batsman Record Comparater',xaxis={'title':'Season'},yaxis={'title':'Runs'})

    fig=go.Figure(data=data,layout=layout)

    pyo.plot(fig,filename='year_by_year.html')

batsman_comp('V Kohli','MS Dhoni','S Dhawan','DA Warner')

# 3. Bar Plot

In [121]:
top10=ipl.groupby('batter')['batsman_runs'].sum().sort_values(ascending=False).head(10).reset_index()

trace=go.Bar(x=top10['batter'],y=top10['batsman_runs'])

data=[trace]

layout=go.Layout(title='Top 10 IPL Batsman',xaxis={'title':'Batsman'},yaxis={'title':'Runs'})

fig=go.Figure(data=data,layout=layout)

pyo.plot(fig,filename='bar_chart.html')

'bar_chart.html'

# There are 3 type of Bar Chart

1. Nested(Group)(Default)
2. Stacked('Relative)
3. Overlayed

In [112]:
top10_index=top10.set_index('batter').squeeze().index
top10_df=ipl[ipl['batter'].isin(top10_index)]
mask1=top10_df['inning']==1
mask2=top10_df['inning']==2
first_inning=top10_df[mask1].groupby(['batter','inning'])['batsman_runs'].sum().reset_index().rename(columns={'batsman_runs':'First Inning'}).drop(columns='inning')
second_inning=top10_df[mask2].groupby(['batter','inning'])['batsman_runs'].sum().reset_index().rename(columns={'batsman_runs':'Second Inning'}).drop(columns='inning')

final=first_inning.merge(second_inning,on='batter')


In [122]:
# Overlayed
trace1=go.Bar(x=final['batter'],y=final['First Inning'],name='First Inning')
trace2=go.Bar(x=final['batter'],y=final['Second Inning'],name='Second Inning')

data=[trace1,trace2]

layout=go.Layout(title='Inning Wise Scores',xaxis={'title':'Batsman'},yaxis={'title':'Score'},barmode='overlay')

fig=go.Figure(data=data,layout=layout)

pyo.plot(fig,filename='over_layed_barchart.html')

'over_layed_barchart.html'

In [123]:
# Stacked
trace1=go.Bar(x=final['batter'],y=final['First Inning'],name='First Inning')
trace2=go.Bar(x=final['batter'],y=final['Second Inning'],name='Second Inning')

data=[trace1,trace2]

layout=go.Layout(title='Inning Wise Scores',xaxis={'title':'Batsman'},yaxis={'title':'Score'},barmode='stack')

fig=go.Figure(data=data,layout=layout)

pyo.plot(fig,filename='stacked_barchart.html')

'stacked_barchart.html'

In [124]:
# Group
trace1=go.Bar(x=final['batter'],y=final['First Inning'],name='First Inning')
trace2=go.Bar(x=final['batter'],y=final['Second Inning'],name='Second Inning')

data=[trace1,trace2]

layout=go.Layout(title='Inning Wise Scores',xaxis={'title':'Batsman'},yaxis={'title':'Score'},barmode='group')

fig=go.Figure(data=data,layout=layout)

pyo.plot(fig,filename='grouped_barchart.html')

'grouped_barchart.html'

# 4. Bubble Plot

In [12]:
only_sixes=new_ipl[new_ipl['batsman_runs']==6]
new_df['Sixes']=only_sixes.groupby('batter')['batsman_runs'].count().values

new_trace=go.Scatter(x=new_df['avg'],y=new_df['Strike Rate'],mode='markers',
                     marker={'size':new_df['Sixes'],
                             'color':new_df['Sixes'],
                             'colorscale':'viridis',
                             'showscale':True
                             },
                     text=new_df['batter'])

data=[new_trace]

layout=go.Layout(title='Bubble Chart Plot',xaxis={'title':'Avg'},yaxis={'title':'Strike Rate'})

fig=go.Figure(data=data,layout=layout)
pyo.plot(fig,filename='bubblechart.html')


'bubblechart.html'

# 5. Box plot

In [159]:
match_agg=delivery.groupby('match_id')['total_runs'].sum().sort_index().reset_index()
season_wise=match_agg.merge(match,left_on='match_id',right_on='id')[['match_id','total_runs','season']]

trace=go.Box(x=season_wise['total_runs'],name='All Seasons')

data=[trace]

layout=go.Layout(title='Total Score Analysis',xaxis={'title':'Total Runs'})

fig=go.Figure(data=data,layout=layout)

pyo.plot(fig,filename='box_plot.html')

'box_plot.html'

In [165]:
trace=go.Box(x=season_wise[season_wise['season']=='2024']['total_runs'],name='2024')
trace2=go.Box(x=season_wise[season_wise['season']=='2018']['total_runs'],name='2018')

data=[trace,trace2]

layout=go.Layout(title='Total Score Analysis',xaxis={'title':'Total Runs'})

fig=go.Figure(data=data,layout=layout)

pyo.plot(fig,filename='box_plot__.html')

'box_plot__.html'

# 6. Distplot

In [14]:
import plotly.figure_factory as ff

hist_data=[new_df['avg'],new_df['Strike Rate']]
group_labels=['Average','Strike Rate']

fig=ff.create_distplot(hist_data=hist_data,group_labels=group_labels)
pyo.plot(fig, filename='distplot.html')

'distplot.html'

# 7. Histogram

In [30]:
x=delivery.groupby('batter')['batsman_runs'].count()>150
x=x[x].index

new=delivery[delivery['batter'].isin(x)]

balls=new.groupby('batter')['batsman_runs'].count()
runs=new.groupby('batter')['batsman_runs'].sum()

sr=(runs/balls)*100
sr=sr.reset_index()
sr

,batter,batsman_runs
0,A Ashish Reddy,142.857143
1,A Badoni,125.544554
2,A Manohar,127.624309
3,A Mishra,86.590909
4,A Symonds,124.711908
...,...,...
235,Y Venugopal Rao,113.872832
236,YBK Jaiswal,146.757991
237,YK Pathan,138.046272
238,YV Takawale,104.918033


In [32]:
trace=go.Histogram(x=sr['batsman_runs'],xbins={'size':2})

data=[trace]
layout=go.Layout(title='Strike Rate Analysis',xaxis={'title':'Strike Rate'})

fig=go.Figure(data,layout)

pyo.plot(fig,filename='hist.html')

'hist.html'

# 8. Heatmap

In [53]:
six=delivery[delivery['batsman_runs']==6]
six=six.groupby(['batting_team','over'])['batsman_runs'].count().reset_index()
six=six.replace({'Royal Challengers Bangalore':'Royal Challengers Bengaluru','Rising Pune Supergiants':'Rising Pune Supergiant',
             'Delhi Daredevils':'Delhi Capitals',
             'Kings XI Punjab':'Punjab Kings'})


In [54]:
trace=go.Heatmap(x=six['batting_team'],y=six['over'],z=six['batsman_runs'])

data=[trace]

layout=go.Layout(title='Total sixes per over analysis',xaxis={'title':'Batting_Team'},yaxis={'title':'Overs'})

fig=go.Figure(data,layout)
pyo.plot(fig,filename='heatmap.html')

'heatmap.html'

# Side by Side Heatmap

In [55]:
dots=delivery[delivery['batsman_runs']==0]
dots=dots.groupby(['batting_team','over'])['batsman_runs'].count().reset_index()
dots=dots.replace({'Royal Challengers Bangalore':'Royal Challengers Bengaluru','Rising Pune Supergiants':'Rising Pune Supergiant',
             'Delhi Daredevils':'Delhi Capitals',
             'Kings XI Punjab':'Punjab Kings'})

In [56]:
from plotly import tools

trace1=go.Heatmap(x=six['batting_team'],y=six['over'],z=six['batsman_runs'].values)
trace2=go.Heatmap(x=dots['batting_team'],y=dots['over'],z=dots['batsman_runs'].values)

fig=tools.make_subplots(rows=1,cols=2,subplot_titles=['6s','0s'],shared_yaxes=True)

fig.append_trace(trace1,1,1)
fig.append_trace(trace2,1,2)

pyo.plot(fig,filename='side_heatmap.html')

d:\MUET\AI-ML Journey\AI Engineering\Data Visualization\env\Lib\site-packages\plotly\tools.py:445: DeprecationWarning: plotly.tools.make_subplots is deprecated, please use plotly.subplots.make_subplots instead
  warnings.warn(


'side_heatmap.html'